# XGBoost Patient Risk Model Training

This notebook focuses on training an XGBoost model for patient risk prediction, performing hyperparameter tuning using Optuna, and explaining the model predictions using SHAP.

In [1]:
import pandas as pd
import numpy as np
import optuna
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split
from src.models.xgboost_trainer import train_xgboost_model
from src.models.hyperparameter_tuner import tune_hyperparameters
from src.models.explainability import explain_model

# Load dataset
data = pd.read_csv('data/patient_data.csv')

# Preprocess data
X = data.drop(columns=['risk_label'])
y = data['risk_label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Hyperparameter tuning
study = optuna.create_study(direction='maximize')
study.optimize(tune_hyperparameters, n_trials=50)

# Train the model with the best parameters
best_params = study.best_params
model = train_xgboost_model(X_train, y_train, best_params)

# Evaluate the model
accuracy = model.score(X_test, y_test)
print(f'Model accuracy: {accuracy:.2f}')

# Explain the model predictions
explainer = shap.Explainer(model)
shap_values = explainer(X_test)
shap.summary_plot(shap_values, X_test)

# Save the model
model.save_model('models/xgboost_patient_risk_model.json')

# Save the study for future reference
with open('models/hyperparameter_study.pkl', 'wb') as f:
    pickle.dump(study, f)
